# 01 — Setup, model loading, and a first feature-extraction tour

**Project:** [EmoVecLLM](https://github.com/drgzkr/EmoVecLLM) — replicating Anthropic 2026 emotion vectors on open-source LLMs.

**This notebook walks through the smallest possible end-to-end loop:**

1. Install dependencies and pick a device (Colab T4/A100 or local).
2. Load a small Transformer (`gpt2`, 124 M) via [TransformerLens](https://transformerlensorg.github.io/TransformerLens/).
3. Tokenize a single prompt and run a forward pass with caching.
4. Extract per-layer × per-token activations as NumPy arrays — both the **cumulative residual stream** (the running state) and the **per-block contribution** (the layer-wise *innovation*, distinct from the residual).
5. **CCN-style geometry** — token RDM, cross-layer similarity matrix, and per-token PCA trajectories through residual-stream space.
6. Pull attention patterns at one layer (head grid) and across all layers (depth evolution, head-averaged).
7. Highlight which previous token each head attends to from the final position.
8. Save residuals, block contributions, attentions, and derived geometry summaries to `data/processed/` for downstream notebooks.

**Runtime.** ~30 s end-to-end on free Colab T4; ~1–2 min on CPU.

**Why GPT-2 small?** Zero auth, downloads in seconds, runs anywhere. We will move to Pythia-1.4B / Llama-3.1-8B in notebook 02 once the workflow is validated.

## 0. Colab / local setup

Installs the only dependency that does not ship with Colab by default. If running locally inside the project venv (`pip install -e .`), the install is a no-op.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Lightweight install — only what this notebook needs.
    !pip install -q transformer-lens

    # HF token from Colab Secrets (only needed for gated models like Llama-3).
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        if token:
            os.environ["HF_TOKEN"] = token
    except Exception:
        pass

    # Persistent state — uncomment to write outputs to Drive.
    # from google.colab import drive; drive.mount("/content/drive")
    # WORK_DIR = "/content/drive/MyDrive/EmoVecLLM"
    WORK_DIR = "/content/EmoVecLLM"
else:
    WORK_DIR = os.path.expanduser("~/Projects/EmoVecLLM")

os.makedirs(os.path.join(WORK_DIR, "data", "processed"), exist_ok=True)
print(f"WORK_DIR = {WORK_DIR}")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from transformer_lens import HookedTransformer

torch.manual_seed(0)
np.random.seed(0)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "font.size": 9,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
})

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device = {device}")

## 1. Load a model

TransformerLens's `HookedTransformer` is a thin wrapper that exposes named hooks at every component (residual stream, attention pattern, MLP, etc.). We will use those hooks directly to pull activations and, in later notebooks, to inject steering vectors.

In [ ]:
MODEL_NAME = "gpt2"   # 124 M params. Swap for "EleutherAI/pythia-1.4b" once validated.

model = HookedTransformer.from_pretrained(MODEL_NAME, device=device)
model.eval()

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
d_model = model.cfg.d_model
print(f"{MODEL_NAME}: {n_layers} layers × {n_heads} heads, d_model = {d_model}")

## 2. Pick a prompt and tokenize

We use a short narrative sentence with mild affective content. The prompt is small enough that the attention heatmaps are legible; large enough that there is real structure to look at. Tokens are byte-pair, so leading whitespace becomes part of the token string (`' heard'` ≠ `'heard'`).

In [ ]:
prompt = "She heard the door creak open and her heart began to race"

token_strs = model.to_str_tokens(prompt)
n_tokens = len(token_strs)
print(f"{n_tokens} tokens:")
for i, t in enumerate(token_strs):
    print(f"  {i:2d}  {t!r}")

## 3. Forward pass with cache → per-layer residual stream

### A 90-second mental model of the residual stream

Picture each token position as a vector that flows **upwards** through the network. The starting point is the token's embedding; the ending point is what gets multiplied by the unembedding matrix to produce the next-token logits. In between, every transformer block (attention + MLP) *reads* the current vector, computes some update, and *adds* the update back in:

$$\text{resid}_{i+1, t} \;=\; \text{resid}_{i, t} \;+\; \text{attn}_i(\text{resid}_i)_t \;+\; \text{mlp}_i(\text{resid}_i)_t$$

Two consequences fall out:

- **It's one shared workspace, not a pipeline of bottlenecks.** Every component reads from and writes to the same stream, so the stream behaves like a high-bandwidth bus that all blocks can listen on (Elhage et al. 2021, *A Mathematical Framework for Transformer Circuits*).
- **Information accumulates additively.** Nothing is overwritten — only added. That's what makes *linear* directions in the residual stream — like the 171 emotion vectors we'll compute in notebook 05 — so useful: adding `scale × vec` at one layer literally injects that direction into every subsequent block's input. It's the same intervention Anthropic uses to shift blackmail rate from 22 % → 72 %.

### Pulling it out

`run_with_cache` runs a forward pass and stashes every named activation. The two we want here:

- `blocks.{i}.hook_resid_post` — residual stream **after** block *i* has finished, shape `(seq_len, d_model)` once the batch dim is stripped.
- `blocks.{i}.attn.hook_pattern` — softmaxed attention weights inside block *i*, shape `(n_heads, seq_q, seq_k)`.

We stack residuals into a single NumPy array `resid` of shape `(n_layers, seq_len, d_model)`. Each row `resid[i]` is a snapshot of the workspace after layer *i* has had its turn.

In [ ]:
with torch.no_grad():
    _, cache = model.run_with_cache(prompt, remove_batch_dim=True)

resid = np.stack(
    [cache[f"blocks.{i}.hook_resid_post"].cpu().float().numpy() for i in range(n_layers)],
    axis=0,
)
embed = cache["hook_embed"].cpu().float().numpy()

print(f"resid.shape = {resid.shape}   # (n_layers, seq_len, d_model)")
print(f"embed.shape = {embed.shape}   # (seq_len, d_model)")

## 4. Cumulative view — residual-stream norm, z-scored across layers

A single heatmap of `‖resid[i, t]‖₂` over layers (y) and tokens (x). Because every block keeps adding to the stream, the raw norm grows monotonically with depth — that gradient swamps any per-token structure. We **z-score each token's norm timeseries across layers** (subtract per-token mean, divide by per-token std), the same trick fMRI uses on voxel timeseries. What's left is depth dynamics relative to each token's own baseline: red = layer where the token is unusually *more* active than its own average; blue = unusually *less*. Spikes flag tokens the network puts disproportionate effort into representing at specific depths.

In [ ]:
resid_norm = np.linalg.norm(resid, axis=-1)   # (n_layers, seq_len)

# Z-score each token's norm timeseries across layers — removes the
# monotonic depth gradient (every block writes additively into the stream,
# so raw norms grow with layer) and leaves only per-token depth dynamics.
mu = resid_norm.mean(axis=0, keepdims=True)
sigma = resid_norm.std(axis=0, keepdims=True) + 1e-12
z = (resid_norm - mu) / sigma

fig, ax = plt.subplots(figsize=(0.5 * n_tokens + 2, 0.3 * n_layers + 1.5))
vmax = float(np.max(np.abs(z)))
im = ax.imshow(z, aspect="auto", cmap="RdBu_r",
               vmin=-vmax, vmax=vmax, origin="lower")
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(token_strs, rotation=45, ha="right")
ax.set_yticks(range(n_layers))
ax.set_ylabel("layer")
ax.set_title("‖residual stream‖₂  (z-scored per token across layers)")
fig.colorbar(im, ax=ax, shrink=0.8, label="z-score")
fig.tight_layout()
plt.show()

## 4b. Block contribution — what *each* layer wrote (distinct from the cumulative residual)

The §4 heatmap shows the **cumulative** residual stream — the running state after every block has had its turn. But each block's *individual* write to that stream is a separate, meaningful object. By the residual-stream identity:

$$\Delta_i \;\equiv\; \text{resid}_i - \text{resid}_{i-1} \;=\; \text{attn}_i(\text{resid}_{i-1}) \;+\; \text{mlp}_i(\text{resid}_{i-1})$$

Compare the two views:

- **Residual norm** (§4) tells you *"what does the model think about token *t* after layer *i*?"* — the cumulative state.
- **Block-contribution norm** (here) tells you *"how much did layer *i* update its thinking about token *t*?"* — the per-layer innovation.

In CCN terms: the residual is the BOLD-like state vector accumulating across depth; the block contribution is the layer-wise *delta* — the analogue of an impulse at each "timepoint" along the depth axis. Bright cells flag positions where a block did unusually heavy work; dark cells are layers that mostly let the stream pass through unchanged.

In [ ]:
# Block contribution = how much this layer ADDED to the stream.
#   block_contrib[0] = resid[0] - embed
#   block_contrib[i] = resid[i] - resid[i-1]   for i >= 1
prev = np.concatenate([embed[None, :, :], resid[:-1]], axis=0)   # (n_layers, seq, d)
block_contrib = resid - prev                                       # (n_layers, seq, d)
contrib_norm = np.linalg.norm(block_contrib, axis=-1)              # (n_layers, seq)

fig, ax = plt.subplots(figsize=(0.5 * n_tokens + 2, 0.3 * n_layers + 1.5))
im = ax.imshow(contrib_norm, aspect="auto", cmap="magma", origin="lower")
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(token_strs, rotation=45, ha="right")
ax.set_yticks(range(n_layers))
ax.set_ylabel("layer  i")
ax.set_title(r"Block contribution  $\|\mathrm{resid}_i - \mathrm{resid}_{i-1}\|_2$  per layer × token")
fig.colorbar(im, ax=ax, shrink=0.8, label="L2 norm")
fig.tight_layout()
plt.show()

## 5. Representational geometry — an RSA / PCA tour

A short detour through standard cognitive-neuroscience tooling, treating each layer as an "ROI" and the tokens as its stimuli. Five views:

- **(a) Token RDM** at one mid-stack layer — Kriegeskorte's classic representational dissimilarity matrix (`1 − cosine`). Stimuli that the model represents alike sit close on the diagonal-adjacent off-diagonal; semantically distant tokens light up far from it.
- **(b) Cross-layer similarity matrix** — for every pair of layers, the token-averaged cosine similarity between matching token representations. Bright bands along the diagonal mean adjacent layers preserve geometry; sharp transitions flag layers that *rewrite* it.
- **(c) PCA — two threadings of the same fit.** Same `(layer, position)` residuals projected to PC1/PC2, plotted twice. **Left** threads along *depth* (lines connect layers within a token — within-pass evolution); **right** threads along *context* (lines connect positions within a layer — across-context evolution as the prefix-summary walks through embedding space with each new token). Causal masking guarantees that the residual at position *t* depends only on tokens 0…*t*, so a single forward pass with the full prompt is equivalent to *N* runs with progressively longer prefixes; the two threadings expose those two complementary views of the same tensor.
- **(d) Per-layer multivariate timeseries — residual stream.** Treat the residual at each layer as a `(d_model, n_tokens)` matrix — features × cumulative-context timepoints, exactly the shape of an fMRI ROI's `(voxels, TRs)` table. Two figures: (i) raw feature × time heatmaps (z-scored per feature for visual contrast), and (ii) timepoint × timepoint **Pearson correlation** matrices — the LLM analogue of a per-layer functional-connectivity matrix.
- **(e) Same views, but for the per-layer *components* of the residual stream — attention output, MLP output, MLP post-activation.** This is where you can localise *what kind of computation* (routing vs feature-transformation) is responsible for the temporal structure each layer's residual carries.

In [ ]:
# (a) Token RDM at one mid-stack layer.

def cosine_rdm(X):
    """X: (n_items, n_features). Returns 1 - cosine_sim, shape (n_items, n_items)."""
    Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
    return 1.0 - Xn @ Xn.T

L = n_layers // 2
rdm = cosine_rdm(resid[L])   # (n_tokens, n_tokens)

fig, ax = plt.subplots(figsize=(0.4 * n_tokens + 2.5, 0.4 * n_tokens + 1.8))
im = ax.imshow(rdm, cmap="viridis", vmin=0, vmax=float(np.percentile(rdm, 99)))
ax.set_xticks(range(n_tokens))
ax.set_xticklabels(token_strs, rotation=45, ha="right")
ax.set_yticks(range(n_tokens))
ax.set_yticklabels(token_strs)
ax.set_title(f"Token RDM (1 − cosine)  —  layer {L}")
fig.colorbar(im, ax=ax, shrink=0.8, label="dissimilarity")
fig.tight_layout()
plt.show()

In [ ]:
# (b) Cross-layer similarity matrix — token-averaged cosine between matching tokens.

resid_n = resid / (np.linalg.norm(resid, axis=-1, keepdims=True) + 1e-12)
cross_sim = np.einsum("ltd,mtd->lm", resid_n, resid_n) / n_tokens   # (n_layers, n_layers)

fig, ax = plt.subplots(figsize=(0.3 * n_layers + 3, 0.3 * n_layers + 2.5))
im = ax.imshow(cross_sim, cmap="RdBu_r", vmin=-1, vmax=1, origin="lower")
ax.set_xlabel("layer  j")
ax.set_ylabel("layer  i")
ax.set_xticks(range(n_layers))
ax.set_yticks(range(n_layers))
ax.set_title("Cross-layer representational similarity\n(token-averaged cosine, residual stream)")
fig.colorbar(im, ax=ax, shrink=0.8, label="cosine")
fig.tight_layout()
plt.show()

In [ ]:
# (c) Two threadings of the same PCA — depth axis vs context axis.
#
# Same fitted PCA over every (layer, position) residual, plotted twice:
#   LEFT  — lines connect layers within a token   → within-pass / depth evolution
#   RIGHT — lines connect positions within a layer → across-context evolution
#
# Causal masking makes these two views internally consistent: position t's
# residual at layer i is exactly what you'd get from running the model on
# prompt[:t+1] (no information from later tokens leaks back). So the right
# panel = how the prefix-summary walks at each depth as more tokens arrive;
# the left panel = how each token's representation evolves through the stack.

flat = resid.reshape(n_layers * n_tokens, d_model)
pca = PCA(n_components=2).fit(flat)
proj = pca.transform(flat).reshape(n_layers, n_tokens, 2)

cmap = plt.cm.viridis

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharex=True, sharey=True)

# Shared scatter — every (layer, position) point, colour = layer.
for ax in axes:
    for L_ in range(n_layers):
        ax.scatter(proj[L_, :, 0], proj[L_, :, 1],
                   color=cmap(L_ / max(1, n_layers - 1)),
                   s=22, alpha=0.85, edgecolor="white", linewidth=0.4, zorder=1)
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")

# LEFT — depth threading: one line per token, connecting layers in order.
ax = axes[0]
for t in range(n_tokens):
    ax.plot(proj[:, t, 0], proj[:, t, 1],
            color="grey", alpha=0.25, lw=0.7, zorder=0)
for t in range(n_tokens):
    ax.annotate(token_strs[t].strip(), (proj[-1, t, 0], proj[-1, t, 1]),
                fontsize=6, alpha=0.85,
                xytext=(3, 3), textcoords="offset points")
ax.set_title("Depth threading — lines = layers within a token\n(within-pass evolution)")

# RIGHT — context threading: one line per layer, connecting positions in order.
ax = axes[1]
for L_ in range(n_layers):
    ax.plot(proj[L_, :, 0], proj[L_, :, 1],
            color=cmap(L_ / max(1, n_layers - 1)),
            alpha=0.55, lw=0.9, zorder=0)
# Anchor the start/end positions at the top layer for orientation.
ax.annotate(f"t=0  {token_strs[0].strip()!r}",
            (proj[-1, 0, 0], proj[-1, 0, 1]),
            fontsize=7, color="#222",
            xytext=(5, 5), textcoords="offset points")
ax.annotate(f"t={n_tokens - 1}  {token_strs[-1].strip()!r}",
            (proj[-1, -1, 0], proj[-1, -1, 1]),
            fontsize=7, color="#222",
            xytext=(5, 5), textcoords="offset points")
ax.set_title("Context threading — lines = positions within a layer\n(across-context evolution)")

# Single shared colorbar for layer index.
sm = plt.cm.ScalarMappable(cmap=cmap,
                           norm=plt.Normalize(vmin=0, vmax=n_layers - 1))
sm.set_array([])
fig.colorbar(sm, ax=axes, label="layer index", shrink=0.9, pad=0.02)

fig.suptitle("Two axes of evolution — same PCA, different threading", y=1.02)
plt.show()

### 5d. Per-layer multivariate timeseries — features × time, paired with timepoint correlation

Two views that take the residual stream at each layer seriously as a **multivariate timeseries**: `d_model` "feature channels" × `n_tokens` cumulative-context timepoints. Same `(voxels, TRs)` shape an fMRI ROI gives you.

Layout: **one row per layer**, with the feature × time heatmap on the left and the timepoint × timepoint Pearson correlation matrix on the right (`width_ratios=[3, 1]`, so both panels read the same per-token x-axis spacing). Reading across compares a layer's raw multivariate trace against its temporal-structure summary at a glance.

- **Left (feature × time)** — `imshow` of the `(d_model, n_tokens)` matrix, **z-scored per feature** across tokens (so each feature's deviation from its own mean is visible on a comparable scale). Vertical bands light up tokens that drive many features at once; horizontal bands flag features that are active across many positions.
- **Right (timepoint × timepoint correlation)** — Pearson correlation across the 768 feature channels for every pair of token positions, equal-aspect square. The LLM analogue of a per-layer functional-connectivity matrix. Bright off-diagonal blocks flag prefixes whose representational *patterns* align; cool blocks flag transitions.

Sharp differences across layers tell you which depth regimes carry which kind of temporal structure.

In [ ]:
# (d) Per-layer multivariate timeseries paired with the timepoint correlation.
#
# Layout: one row per layer; left = (d_model, n_tokens) feature × time heatmap,
# right = (n_tokens, n_tokens) Pearson correlation across the d_model dimension.
# width_ratios=[3, 1] so the time axis on both panels reads at the same
# pixels-per-token spacing, with the corr matrix square (aspect="equal").
#
# `multivar_views` is defined here and re-used by §5e for the components.

def time_corr(X):
    """X: (d_model, n_tokens). Returns (n_tokens, n_tokens) Pearson correlation."""
    Xc = X - X.mean(axis=0, keepdims=True)
    Xn = Xc / (Xc.std(axis=0, keepdims=True) + 1e-12)
    return (Xn.T @ Xn) / X.shape[0]


def multivar_views(act, name):
    """Per-layer rows: features × time (left) | timepoint × timepoint corr (right)."""
    n_layers_, n_tokens_, d_feat = act.shape

    # Z-score per (layer, feature) across tokens.
    mu_a = act.mean(axis=1, keepdims=True)
    sigma_a = act.std(axis=1, keepdims=True) + 1e-12
    act_z = (act - mu_a) / sigma_a
    vmax_z = float(np.percentile(np.abs(act_z), 99))

    # Pearson correlation across the feature dim per layer.
    corr_all = np.stack(
        [time_corr(act[L_].T) for L_ in range(n_layers_)],
        axis=0,
    )

    fig, axes = plt.subplots(
        n_layers_, 2,
        figsize=(11, 2.6 * n_layers_),
        gridspec_kw={"width_ratios": [3, 1]},
        constrained_layout=True,
    )
    if n_layers_ == 1:
        axes = axes[None, :]

    for L_ in range(n_layers_):
        ax_ts, ax_cm = axes[L_, 0], axes[L_, 1]

        ax_ts.imshow(act_z[L_].T, aspect="auto", cmap="RdBu_r",
                     vmin=-vmax_z, vmax=vmax_z, interpolation="nearest")
        ax_ts.set_title(f"layer {L_}  —  features × tokens  (z-scored, ±{vmax_z:.1f})",
                        fontsize=9, loc="left")
        ax_ts.set_xticks(range(n_tokens_))
        ax_ts.set_xticklabels(token_strs, rotation=90, fontsize=6)
        ax_ts.set_ylabel(f"feature ({d_feat})", fontsize=7)

        ax_cm.imshow(corr_all[L_], cmap="RdBu_r", vmin=-1, vmax=1, aspect="equal")
        ax_cm.set_title("timepoint × timepoint  Pearson corr",
                        fontsize=9, loc="left")
        ax_cm.set_xticks(range(n_tokens_))
        ax_cm.set_yticks(range(n_tokens_))
        ax_cm.set_xticklabels(token_strs, rotation=90, fontsize=6)
        ax_cm.set_yticklabels(token_strs, fontsize=6)

    fig.suptitle(name, fontsize=12, y=1.0)
    plt.show()
    return corr_all


# Run for the residual stream.
_ = multivar_views(
    resid,
    "Residual stream — per-layer features × tokens + temporal correlation",
)

### 5e. Same paired layout — but for the per-layer *components* of the stream

§5d treated the residual itself — the *cumulative* state. Each block's *write* to that stream comes from two sub-components: attention and MLP. Splitting them out and re-running the same paired layout (features × tokens on the left, timepoint × timepoint correlation on the right, one row per layer) lets you ask **which component is responsible for the temporal structure each layer's residual carries**:

- **`hook_attn_out`** — attention's write to the residual at this layer. Shape `(n_tokens, d_model)`. Captures *routing* of information across token positions.
- **`hook_mlp_out`** — MLP's write to the residual at this layer. Shape `(n_tokens, d_model)`. Captures *feature transformation* at each position.
- **`mlp.hook_post`** — MLP's internal post-activation state. Shape `(n_tokens, 4 × d_model)` (= `d_mlp`, the wider intermediate). This is where most sparse-autoencoder work operates and where individually-interpretable features tend to be densest.

Reading the comparison: if a layer's residual shows clean temporal structure but its `attn_out` is mostly flat while `mlp_out` carries the structure, the MLP at that layer is doing the work. Reverse pattern → attention is doing the work. Both → joint contribution. This is the standard mech-interp move for localising *what kind of computation* lives at *which depth* — and once we extract emotion vectors in nb05, we will lean on it to pick the layer / component combination that gives the cleanest steering signal.

In [ ]:
# (e) Same paired layout for the per-layer components.
# `multivar_views` was defined in §5d above; we just need to pull each
# component stream from the cache and call it.

attn_out_all = np.stack(
    [cache[f"blocks.{i}.hook_attn_out"].cpu().float().numpy() for i in range(n_layers)],
    axis=0,
)   # (n_layers, n_tokens, d_model)
mlp_out_all = np.stack(
    [cache[f"blocks.{i}.hook_mlp_out"].cpu().float().numpy() for i in range(n_layers)],
    axis=0,
)   # (n_layers, n_tokens, d_model)
mlp_post_all = np.stack(
    [cache[f"blocks.{i}.mlp.hook_post"].cpu().float().numpy() for i in range(n_layers)],
    axis=0,
)   # (n_layers, n_tokens, d_mlp)

print(f"attn_out:  {attn_out_all.shape}")
print(f"mlp_out:   {mlp_out_all.shape}")
print(f"mlp_post:  {mlp_post_all.shape}")

_ = multivar_views(
    attn_out_all,
    "Attention output (hook_attn_out) — features × tokens + temporal correlation",
)
_ = multivar_views(
    mlp_out_all,
    "MLP output (hook_mlp_out) — features × tokens + temporal correlation",
)
_ = multivar_views(
    mlp_post_all,
    "MLP post-activation (mlp.hook_post, d_mlp = 4 × d_model) — features × tokens + temporal correlation",
)

## 6. Attention patterns for one layer — head grid

Each subplot is the `(seq_q, seq_k)` attention matrix for one head, in a single chosen layer — **rows are query tokens (the position doing the looking), columns are key tokens (the positions being looked at)**. Every subplot carries its own token labels so each head can be read directly without cross-referencing edges. Mid-stack layers tend to show the richest mix (induction-style off-diagonals, previous-token heads, BOS heads, etc.). Lower-triangular shape comes from causal masking.

In [ ]:
LAYER = n_layers // 2

attn = cache[f"blocks.{LAYER}.attn.hook_pattern"].cpu().float().numpy()   # (heads, q, k)
print(f"attn.shape = {attn.shape}")

ncols = 4
nrows = int(np.ceil(n_heads / ncols))
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.8 * ncols, 2.8 * nrows + 0.5),
    constrained_layout=True,
)
axes = np.atleast_2d(axes)

for h in range(n_heads):
    ax = axes.flat[h]
    ax.imshow(attn[h], cmap="Blues", vmin=0, vmax=1, aspect="equal")
    ax.set_title(f"L{LAYER} H{h}", fontsize=8)
    ax.set_xticks(range(n_tokens))
    ax.set_yticks(range(n_tokens))
    ax.set_xticklabels(token_strs, rotation=90, fontsize=5)
    ax.set_yticklabels(token_strs, fontsize=5)

for h in range(n_heads, nrows * ncols):
    axes.flat[h].axis("off")

fig.suptitle(
    f"Attention patterns — layer {LAYER}   (rows = query token, columns = key token)",
    fontsize=10,
)
plt.show()

## 6b. Attention evolution across layers

§6 showed every head's attention pattern at *one* layer. To see how attention restructures with depth, we collapse over heads (mean across the head axis) and plot the head-averaged `(seq_q, seq_k)` matrix at *every* layer.

Layer-to-layer changes in the diagonal / off-diagonal pattern flag where the model is rewiring its routing of information. Early layers tend to look "local" (strong diagonal-1 / previous-token heads averaging in); mid-layers add longer-range dependencies; late layers often concentrate attention onto a few "summary" positions or onto the BOS token.

In [ ]:
# Stack all layers' attention, then average over heads.
attn_all = np.stack(
    [cache[f"blocks.{i}.attn.hook_pattern"].cpu().float().numpy() for i in range(n_layers)],
    axis=0,
)   # (n_layers, n_heads, seq_q, seq_k)
attn_layer_avg = attn_all.mean(axis=1)   # (n_layers, seq_q, seq_k)

ncols = 4
nrows = int(np.ceil(n_layers / ncols))
fig, axes = plt.subplots(
    nrows, ncols,
    figsize=(2.6 * ncols, 2.7 * nrows + 0.5),
    constrained_layout=True,
)
axes = np.atleast_2d(axes)

vmax = float(attn_layer_avg.max())
for L_ in range(n_layers):
    ax = axes.flat[L_]
    ax.imshow(attn_layer_avg[L_], cmap="Blues", vmin=0, vmax=vmax, aspect="equal")
    ax.set_title(f"layer {L_}", fontsize=8)
    ax.set_xticks(range(n_tokens))
    ax.set_yticks(range(n_tokens))
    ax.set_xticklabels(token_strs, rotation=90, fontsize=5)
    ax.set_yticklabels(token_strs, fontsize=5)

for L_ in range(n_layers, nrows * ncols):
    axes.flat[L_].axis("off")

fig.suptitle(
    f"Attention evolution across layers (mean over {n_heads} heads, rows=query, cols=key)",
    fontsize=10,
)
plt.show()

## 7. Highlight — where does the *final* token look?

For each head in the same layer, the row of attention weights from the last query position. Three views:

- A `(heads × keys)` heatmap with token strings on the x-axis. The **argmax** of each row (the token that head attends to most strongly) is **circled in red**, so you can read the strongest target without scanning intensities.
- A horizontal bar of per-head **attention entropy** (in nats). Low entropy = head concentrates on one or two tokens; high entropy = head spreads attention diffusely.
- A per-head **top-target bar plot** with the actual token *text* labelled next to each bar — the most direct way to see which previous token each head is reading from.

In [ ]:
final_q = n_tokens - 1
attn_from_last = attn[:, final_q, :]   # (n_heads, seq_len)

eps = 1e-12
entropy = -(attn_from_last * np.log(attn_from_last + eps)).sum(axis=-1)

top_k = attn_from_last.argmax(axis=-1)                   # (n_heads,)
top_w = attn_from_last[np.arange(n_heads), top_k]        # (n_heads,)

# Views 1 + 2: heatmap with argmax circled, plus per-head entropy bar.
fig, (ax1, ax2) = plt.subplots(
    1, 2,
    figsize=(0.4 * n_tokens + 6, 0.25 * n_heads + 1.8),
    gridspec_kw={"width_ratios": [3, 1]},
    constrained_layout=True,
)
im = ax1.imshow(attn_from_last, aspect="auto", cmap="Blues", vmin=0, vmax=1)
ax1.set_xticks(range(n_tokens))
ax1.set_xticklabels(token_strs, rotation=45, ha="right")
ax1.set_yticks(range(n_heads))
ax1.set_ylabel("head")
ax1.set_title(f"Attention from final token  —  layer {LAYER}")
ax1.scatter(top_k, np.arange(n_heads),
            marker="o", s=44, facecolor="none",
            edgecolor="#cc3333", linewidth=1.4)
fig.colorbar(im, ax=ax1, shrink=0.8, label="weight")

ax2.barh(range(n_heads), entropy, color="#4477aa")
ax2.set_yticks(range(n_heads))
ax2.invert_yaxis()
ax2.set_xlabel("entropy (nats)")
ax2.set_title("per-head\nentropy")
plt.show()

# View 3: per-head top-attended token, shown as bar + token-text label.
fig, ax = plt.subplots(figsize=(7, 0.3 * n_heads + 1.5))
ax.barh(range(n_heads), top_w, color="#4477aa")
ax.set_yticks(range(n_heads))
ax.set_yticklabels([f"H{h}" for h in range(n_heads)])
ax.invert_yaxis()
ax.set_xlim(0, max(float(top_w.max()) * 1.45, 0.25))
ax.set_xlabel(f"attention weight to top-attended token (layer {LAYER})")
ax.set_title("Each head's top-attended token (text)")
for h in range(n_heads):
    ax.text(top_w[h] + 0.005, h,
            f"{token_strs[top_k[h]]!r}",
            va="center", fontsize=9, color="#222222")
fig.tight_layout()
plt.show()

## 8. Save features to disk

Stack attentions across all layers and save residuals + attentions + the geometry summaries (RDM, cross-layer matrix, PCA projection) as a single `.npz`. Notebook 02 will load this file and start building the actual emotion-vector pipeline on top.

In [ ]:
# attn_all and attn_layer_avg already computed in §6b; reuse.
out = os.path.join(WORK_DIR, "data", "processed", "nb01_features.npz")
np.savez(
    out,
    resid=resid,
    embed=embed,
    block_contrib=block_contrib,
    block_contrib_norm=contrib_norm,
    attn=attn_all,
    attn_layer_avg=attn_layer_avg,
    tokens=np.array(token_strs, dtype=object),
    rdm_mid_layer=rdm,
    rdm_layer_index=L,
    cross_layer_sim=cross_sim,
    pca_proj=proj,
    pca_explained_variance_ratio=pca.explained_variance_ratio_,
    model_name=MODEL_NAME,
)
print(f"Saved → {out}")
print(f"  resid:           {resid.shape}")
print(f"  embed:           {embed.shape}")
print(f"  block_contrib:   {block_contrib.shape}")
print(f"  attn:            {attn_all.shape}")
print(f"  attn_layer_avg:  {attn_layer_avg.shape}")
print(f"  rdm (layer {L}): {rdm.shape}")
print(f"  cross_layer_sim: {cross_sim.shape}")
print(f"  pca_proj:        {proj.shape}")

---

## Next steps

- **Notebook 02 (`02_emotion_word_list_and_prompts.ipynb`)** — source the 171-emotion list and build paired emotion / neutral story prompts.
- **Adapter wrap** — promote the `from_pretrained` + `run_with_cache` pattern used here into `src/emovecllm/models.TransformerLensAdapter`.
- **Try a bigger model** — re-run with `MODEL_NAME = "EleutherAI/pythia-1.4b"` once you confirm the workflow on `gpt2`. On free Colab T4 this still fits comfortably in fp16.
- **More CCN-style views to add later** — per-layer RDM grids (one panel per layer), RSA against external reference RDMs (e.g. valence/arousal-derived target RDMs from Warriner norms), MDS / t-SNE alternatives to PCA, and an early encoding-model regression sketch (predict each layer's residual from token-embedding features, score by R² across layers — directly prefigures the brain-bridge stage).